In [1]:
import requests
import json
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()

# Get secret variables from env file
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY")

# OpenMetro Data

In [2]:
# --- STEP 1: Define what we're asking for ---
# Kamloops, BC — one of the most fire-prone areas in the province.
latitude = 50.67
longitude = -120.33

# --- STEP 2: Build the API URL ---
# Open-Meteo's /v1/forecast endpoint provides weather variables, NOT FWI components.
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": ",".join([
        "temperature_2m",          # Air temp at 2m (°C) — FWI input
        "relative_humidity_2m",    # RH at 2m (%) — FWI input
        "wind_speed_10m",          # Wind at 10m (km/h) — FWI input
        "precipitation",           # Rain in mm — FWI input
        # These four variables are the inputs to the Canadian FWI System.
        # You'll calculate FFMC, DMC, DC, ISI, BUI, and FWI from them."
        "soil_temperature_0cm", 
        "soil_temperature_6cm",
        "soil_temperature_18cm",
        "soil_temperature_54cm"

    ]),
    "timezone": "America/Vancouver",  # Get timestamps in BC local time
}

# --- STEP 3: Make the request ---
response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code != 200:
    print(f"Error: {response.text}")
    exit()

data = response.json()

# --- STEP 4: Convert the data to a DataFrame ---

df = pd.DataFrame(data['hourly'])
print(df.head())
print(df.info())
print(df.describe())


Status code: 200
               time  temperature_2m  relative_humidity_2m  wind_speed_10m  \
0  2026-09-18T00:00            13.0                    52            10.9   
1  2026-09-18T01:00            12.3                    52            10.2   
2  2026-09-18T02:00            11.7                    51            10.5   
3  2026-09-18T03:00            11.2                    52            10.5   
4  2026-09-18T04:00            10.3                    62            11.5   

   precipitation  soil_temperature_0cm  soil_temperature_6cm  \
0            0.0                  10.7                  14.5   
1            0.0                  10.1                  13.8   
2            0.0                   9.6                  13.3   
3            0.0                   9.1                  12.8   
4            0.0                   8.8                  12.4   

   soil_temperature_18cm  soil_temperature_54cm  
0                   16.9                   17.1  
1                   16.7           

In [3]:
# --- STEP 4: Check for API-level errors ---
# Open-Meteo returns {"error": true, "reason": "..."} for bad variable names.
# This is what tripped us up before — we asked for variables it doesn't have.
if data.get("error"):
    print(f"API error: {data.get('reason')}")
    exit()

# --- STEP 5: Understand the response structure ---
print("\n--- TOP-LEVEL KEYS ---")
print(list(data.keys()))

print("\n--- METADATA ---")
print(f"  Requested:  ({latitude}, {longitude})")
print(f"  Snapped to: ({data['latitude']}, {data['longitude']})")
print(f"  Elevation:  {data.get('elevation', 'N/A')} m")
print(f"  Timezone:   {data.get('timezone', 'N/A')}")

print("\n--- HOURLY VARIABLES RETURNED ---")
hourly = data["hourly"]
print(f"  Variables: {[k for k in hourly.keys() if k != 'time']}")
print(f"  Time steps: {len(hourly['time'])}")

# --- STEP 6: Look at actual values (first 24 hours) ---
print("\n--- FIRST 24 HOURS OF DATA ---")
print(f"{'Time':<22} {'Temp°C':>7} {'RH%':>5} {'Wind km/h':>10} {'Precip mm':>10}")
print("-" * 60)

for i in range(min(24, len(hourly["time"]))):
    temp = hourly['temperature_2m'][i]
    rh   = hourly['relative_humidity_2m'][i]
    wind = hourly['wind_speed_10m'][i]
    prec = hourly['precipitation'][i]
    print(
        f"{hourly['time'][i]:<22}"
        f"{temp if temp is not None else 'N/A':>7}"
        f"{rh if rh is not None else 'N/A':>5}"
        f"{wind if wind is not None else 'N/A':>10}"
        f"{prec if prec is not None else 'N/A':>10}"
    )

# --- STEP 7: Quick stats ---
temps = [v for v in hourly["temperature_2m"] if v is not None]
rhs   = [v for v in hourly["relative_humidity_2m"] if v is not None]
winds = [v for v in hourly["wind_speed_10m"] if v is not None]
precs = [v for v in hourly["precipitation"] if v is not None]

print("\n--- VALUE RANGES ---")
print(f"  Temp:   {min(temps):.1f} to {max(temps):.1f} °C")
print(f"  RH:     {min(rhs):.0f} to {max(rhs):.0f} %")
print(f"  Wind:   {min(winds):.1f} to {max(winds):.1f} km/h")
print(f"  Precip: {min(precs):.1f} to {max(precs):.1f} mm")
print(f"\n  Total forecast hours: {len(hourly['time'])}")
print(f"  Null count check: temp={sum(1 for v in hourly['temperature_2m'] if v is None)}, "
      f"rh={sum(1 for v in hourly['relative_humidity_2m'] if v is None)}")



--- TOP-LEVEL KEYS ---
['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']

--- METADATA ---
  Requested:  (50.67, -120.33)
  Snapped to: (50.678204, -120.34752)
  Elevation:  386.0 m
  Timezone:   America/Vancouver

--- HOURLY VARIABLES RETURNED ---
  Variables: ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation', 'soil_temperature_0cm', 'soil_temperature_6cm', 'soil_temperature_18cm', 'soil_temperature_54cm']
  Time steps: 168

--- FIRST 24 HOURS OF DATA ---
Time                    Temp°C   RH%  Wind km/h  Precip mm
------------------------------------------------------------
2026-09-18T00:00         13.0   52      10.9       0.0
2026-09-18T01:00         12.3   52      10.2       0.0
2026-09-18T02:00         11.7   51      10.5       0.0
2026-09-18T03:00         11.2   52      10.5       0.0
2026-09-18T04:00         10.3   62      11.5       0.0
2026-09-18T05:00     

# FIRMS Data

In [4]:
import requests
import pandas as pd
from io import StringIO

# --- STEP 1: Build the URL ---
# Format: /csv/{MAP_KEY}/{source}/{bbox}/{days}
# Source: VIIRS_SNPP_NRT = VIIRS sensor on Suomi NPP satellite, near-real-time
# Bbox: lon_min, lat_min, lon_max, lat_max (BC bounding box)
# Days: 1 = last 24 hours of detections
FIRMS_URL = (
    f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{FIRMS_MAP_KEY}/VIIRS_SNPP_NRT/-139,48,-114,60/1"
)

# --- STEP 2: Fetch the data ---
response = requests.get(FIRMS_URL)
print(f"Status code: {response.status_code}")

if response.status_code != 200:
    print(f"Error: {response.text[:500]}")
else:
    # --- STEP 3: Parse CSV into DataFrame ---
    # The response body is raw CSV text, not JSON.
    # StringIO wraps it so pandas can read it like a file.
    df = pd.read_csv(StringIO(response.text))

    print(f"\n--- SHAPE ---")
    print(f"  {len(df)} fire detections in the last 24 hours")
    print(f"  {len(df.columns)} columns")

    print(f"\n--- COLUMNS ---")
    print(df.columns.tolist())

    print(f"\n--- FIRST 5 ROWS ---")
    print(df.head())

    print(f"\n--- DATA TYPES ---")
    print(df.dtypes)

    # --- STEP 4: Key fields for your risk model ---
    print(f"\n--- KEY FIELDS ---")
    print(f"  Lat range:  {df['latitude'].min():.2f} to {df['latitude'].max():.2f}")
    print(f"  Lon range:  {df['longitude'].min():.2f} to {df['longitude'].max():.2f}")

    # Confidence: 'nominal', 'low', 'high' — you'll want to filter on this
    print(f"\n  Confidence distribution:")
    print(df['confidence'].value_counts().to_string())

    # FRP = Fire Radiative Power (MW) — intensity of the fire
    print(f"\n  FRP (Fire Radiative Power):")
    print(f"    min={df['frp'].min():.1f}, max={df['frp'].max():.1f}, "
          f"mean={df['frp'].mean():.1f} MW")

    # Brightness — thermal signature, higher = more intense
    if 'bright_ti4' in df.columns:
        print(f"\n  Brightness (bright_ti4):")
        print(f"    min={df['bright_ti4'].min():.1f}, max={df['bright_ti4'].max():.1f} K")

    # --- STEP 5: Check for things that matter for your pipeline ---
    # How many detections are high confidence? Those are the ones you trust.
    if 'high' in df['confidence'].values:
        high_conf = df[df['confidence'] == 'high']
        print(f"\n  High-confidence detections: {len(high_conf)} / {len(df)}")
    
    # Date/time field — what does the timestamp look like?
    print(f"\n  Time field (acq_date): {df['acq_date'].iloc[0]} (type: {df['acq_date'].dtype})")
    print(f"  Time field (acq_time): {df['acq_time'].iloc[0]} (type: {df['acq_time'].dtype})")

Status code: 400
Error: Invalid MAP_KEY.


-  latitude — for nearest-fire-proximity calculation (ST_Distance)
- longitude — same
- bright_ti4 — thermal intensity in Kelvin, useful for distinguishing large fires from small ones
- frp — fire radiative power in MW, a direct measure of fire intensity you could use to weight proximity scores later
- acq_date — when the detection happened, needed for filtering stale detections
- acq_time — combined with acq_date gives you the exact observation time

In [5]:
FIRMS_COL_KEEP = ['lattude', 'longitude', 'acq_date', 'acq_time', 'frp', 'bright_ti4']
                  

# AppEARS NASA Data

In [9]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API = "https://appeears.earthdatacloud.nasa.gov/api/"
USERNAME = os.getenv("AppEARS_user")
PASSWORD = os.getenv("AppEARS_pass")

# Verify credentials loaded
print(f"Username: '{USERNAME}'")
print(f"Password loaded: {PASSWORD is not None and len(PASSWORD) > 0}")

# AppEEARS login uses HTTP Basic Auth to get a session token
# This is different from a generic Earthdata bearer token
auth_response = requests.post(
    f"{API}login",
    auth=(USERNAME, PASSWORD),
    timeout=30
)

print(f"Status: {auth_response.status_code}")
print(f"Response: {auth_response.text[:300]}")

if auth_response.status_code == 200:
    token = auth_response.json()["token"]
    headers = {"Authorization": f"Bearer {token}"}
    print("\nAuthenticated successfully — this token is valid for 48 hours.")
else:
    print("\nTroubleshooting:")
    print("  1. Go to https://urs.earthdata.nasa.gov/ and confirm you can log in")
    print("  2. Check your .env has no quotes or spaces: AppEARS_user=myname")
    print("  3. Go to Profile → Applications → Authorized Apps")
    print("     Search for 'LP DAAC Cumulus PROD' and approve it if missing")

Username: 'daniel_y'
Password loaded: True
Status: 200
Response: {"token_type": "Bearer", "token": "-AGDD6uHFD8PcSg1aBTOK5qpDSTHsb7GJG7gOqz--a0K7PbyhBTEfcNxHy2yyD-le3o_JVKo80k7m5p5UqsQMg", "expiration": "2026-09-19T23:30:03Z"}


Authenticated successfully — this token is valid for 48 hours.


In [13]:
import os
import requests
import time
import pandas as pd
from io import StringIO
from dotenv import load_dotenv

load_dotenv()

# --- STEP 2: Check available layers in MOD13A1 ---
product = "MOD13A1.061"
layers_response = requests.get(f"{API}product/{product}", headers=headers)

if layers_response.status_code != 200:
    print(f"Product lookup failed: {layers_response.status_code}")
    raise SystemExit()

layers = layers_response.json()

print(f"\n--- AVAILABLE LAYERS in {product} ---")
for layer_name in layers:
    desc = layers[layer_name].get("Description", "N/A")
    print(f"  {layer_name}: {desc}")

# --- STEP 3: Submit a point request ---
task = {
    "task_type": "point",
    "task_name": "wildfire_ndvi_eda",
    "params": {
        "dates": [
            {
                "startDate": "06-01-2025",
                "endDate": "09-30-2025",
            }
        ],
        "layers": [
            {"product": "MOD13A1.061", "layer": "_500m_16_days_NDVI"},
            {"product": "MOD13A1.061", "layer": "_500m_16_days_pixel_reliability"},
        ],
        "coordinates": [
            {
                "id": "kamloops",
                "latitude": "50.67",
                "longitude": "-120.33",
                "category": "BC_fire_prone",
            }
        ],
    },
}

task_response = requests.post(
    f"{API}task", json=task, headers=headers
).json()

print(f"Response: {task_response}")

if "task_id" not in task_response:
    print(f"Task submission failed: {task_response}")
    raise SystemExit()

task_id = task_response["task_id"]
print(f"\n--- TASK SUBMITTED ---")
print(f"  Task ID: {task_id}")
print(f"  Status: {task_response['status']}")

# --- STEP 4: Poll until complete ---
print("\n--- WAITING FOR PROCESSING ---")
while True:
    status_response = requests.get(
        f"{API}task/{task_id}", headers=headers
    ).json()
    status = status_response["status"]
    print(f"  Status: {status}")

    if status == "done":
        break
    elif status == "error":
        print(f"  Task failed: {status_response}")
        raise SystemExit()

    time.sleep(30)

# --- STEP 5: Download results ---
bundle = requests.get(
    f"{API}bundle/{task_id}", headers=headers
).json()

print(f"\n--- FILES AVAILABLE ---")
for f in bundle["files"]:
    print(f"  {f['file_name']} ({f['file_type']}, {f['file_size']} bytes)")

csv_file = next(
    f for f in bundle["files"] if f["file_name"].endswith(".csv")
)
csv_url = f"{API}bundle/{task_id}/{csv_file['file_id']}"
csv_response = requests.get(csv_url, headers=headers)

df = pd.read_csv(StringIO(csv_response.text))

# --- STEP 6: Explore the data ---
print(f"\n--- SHAPE ---")
print(f"  {len(df)} rows, {len(df.columns)} columns")

print(f"\n--- COLUMNS ---")
print(df.columns.tolist())

print(f"\n--- FIRST ROWS ---")
print(df.head(10))

print(f"\n--- DATA TYPES ---")
print(df.dtypes)

# --- STEP 7: Understand NDVI values ---
ndvi_col = [c for c in df.columns if "NDVI" in c and "reliability" not in c.lower()]
if ndvi_col:
    col = ndvi_col[0]
    df["ndvi_actual"] = df[col] / 10000
    print(f"\n--- NDVI VALUES (scaled to -1 to 1) ---")
    print(f"  Column: {col}")
    print(f"  Raw range:    {df[col].min()} to {df[col].max()}")
    print(f"  Actual range: {df['ndvi_actual'].min():.4f} to {df['ndvi_actual'].max():.4f}")
    print(f"  Mean:         {df['ndvi_actual'].mean():.4f}")

rel_col = [c for c in df.columns if "reliability" in c.lower()]
if rel_col:
    print(f"\n--- PIXEL RELIABILITY ---")
    print(f"  (0=good, 1=marginal, 2=snow/ice, 3=cloudy, 4=not processed)")
    for val, count in df[rel_col[0]].value_counts().items():
        print(f"    {val}: {count}")




--- AVAILABLE LAYERS in MOD13A1.061 ---
  _500m_16_days_EVI: 500m 16 days EVI
  _500m_16_days_MIR_reflectance: 500m 16 days MIR reflectance
  _500m_16_days_NDVI: 500m 16 days NDVI
  _500m_16_days_NIR_reflectance: 500m 16 days NIR reflectance
  _500m_16_days_VI_Quality: 500m 16 days VI Quality
  _500m_16_days_blue_reflectance: 500m 16 days blue reflectance
  _500m_16_days_composite_day_of_the_year: 500m 16 days composite day of the year
  _500m_16_days_pixel_reliability: 500m 16 days pixel reliability
  _500m_16_days_red_reflectance: 500m 16 days red reflectance
  _500m_16_days_relative_azimuth_angle: 500m 16 days relative azimuth angle
  _500m_16_days_sun_zenith_angle: 500m 16 days sun zenith angle
  _500m_16_days_view_zenith_angle: 500m 16 days view zenith angle
Response: {'task_id': 'fa454f3c-7023-4f7b-abbd-06e7a90b62dd', 'status': 'pending'}

--- TASK SUBMITTED ---
  Task ID: fa454f3c-7023-4f7b-abbd-06e7a90b62dd
  Status: pending

--- WAITING FOR PROCESSING ---
  Status: pending
  

## MODIS NDVI — EDA Summary & Pipeline Notes

**Product**: MOD13A1.061 (MODIS Terra, 500m, 16-day composite)

**Access**: AppEEARS API (async job-based workflow — submit task, poll, download CSV). Requires NASA Earthdata login + LP DAAC Cumulus PROD authorization. Token valid for 48 hours.

### Columns to Keep

From the 33 columns returned, keep only three:

- `Date` — observation timestamp (every 16 days)
- `MOD13A1_061__500m_16_days_NDVI` — already scaled to -1 to 1 range by AppEEARS (no need to divide by 10000)
- `MOD13A1_061__500m_16_days_pixel_reliability` — numeric quality flag (0=good, 1=marginal, 2=snow/ice, 3=cloudy, 4=not processed)

Drop everything else (VI_Quality bitmasks, reflectance bands, aerosol flags, tile coordinates, shadow/cloud flags). The pixel reliability column already summarizes all quality information into one usable filter.

### Filtering

Only use rows where `pixel_reliability == 0` (good data). In our Kamloops test, 5 of 9 observations were good and 4 were marginal. For a production pipeline, you could include marginal (1) as a fallback if good data is unavailable, but prefer good.

### What the Values Look Like (Kamloops, Jun–Sep 2025)

NDVI ranged from 0.26 to 0.45. The pattern showed spring green-up (~0.44 in May), a mid-summer dip (~0.26 in mid-July, peak drying), then recovery (~0.45 by September). That mid-July dip is exactly the vegetation stress signal the fire risk model needs to capture.

For context, NDVI around 0.2–0.3 indicates sparse or stressed vegetation (grassland, dry open forest). Dense healthy forest would be 0.6–0.8. Kamloops is semi-arid, so these values are expected.

### How It Feeds Into the Risk Model (15% weight)

Raw NDVI alone isn't enough — a cell with NDVI 0.3 might be normal (alpine, grassland) or dangerously low (dense forest that dried out). The pipeline needs to compute **NDVI anomaly**:

1. Build a seasonal baseline: average NDVI per cell per 16-day period across multiple years (e.g., 2015–2024)
2. For each new observation: `anomaly = current_ndvi - baseline_ndvi`
3. A large negative anomaly = vegetation is drier than normal = higher fire risk
4. Normalize the anomaly to 0–100 for the composite score

### Pipeline Integration

- **Fetch frequency**: weekly (the 16-day composite updates slowly, so more frequent fetching is wasteful)
- **Storage**: raw responses to `data/raw/ndvi/YYYY-MM-DD.parquet`, baselines in `data/static/ndvi_baselines.parquet`
- **Latency tolerance**: high — NDVI changes slowly, so stale data (up to 2 weeks old) is acceptable. If AppEEARS is down, keep using the last successful fetch.
- **Scaling note**: AppEEARS tasks are async and can take minutes. For 1,500 grid cells, submit one task with all coordinates rather than 1,500 individual tasks.